# 🛰️ SongHong SAR Monitoring — Tuần 1
## Chuẩn bị dữ liệu & Thiết lập môi trường

**Tác giả:** Vũ Đức Tùng | **Tháng:** 07/2026

**GEE Project:** `crested-library-500309-i2`

---

### Pipeline tổng quan
```
Phần 1: Kiểm tra môi trường GEE
Phần 2: Thu thập Sentinel-1 GRD (2015–2024)
Phần 3: Tiền xử lý (Speckle filter + VV/VH/ratio)
Phần 4: Kiểm tra trực quan (dry vs wet season)
Phần 5: Export GeoTIFF → Google Drive
```

### Cài đặt
```bash
pip install earthengine-api geemap
earthengine authenticate
```

## ⚡ Setup — Import thư viện & Khởi tạo GEE

In [ ]:
import ee
import geemap
import json, os, csv
from datetime import datetime

ee.Initialize(project='crested-library-500309-i2')
print('✅ GEE initialized: crested-library-500309-i2')

# ── Đường dẫn gốc project
PROJECT_ROOT = os.path.normpath(os.path.join(os.getcwd(), '..'))
AOI_PATH = os.path.join(PROJECT_ROOT, 'aoi', 'song_hong_aoi.geojson')
OUT_DIR  = os.path.join(PROJECT_ROOT, 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)

# ── Load AOI
with open(AOI_PATH, 'r', encoding='utf-8') as f:
    aoi_geojson = json.load(f)

aoi_fc       = ee.FeatureCollection(aoi_geojson)
aoi_geometry = aoi_fc.geometry()
print(f'✅ AOI loaded: {AOI_PATH}')
print(f'✅ Outputs  : {OUT_DIR}')

---

## 🔍 Phần 1 — Kiểm tra môi trường GEE

Khởi tạo kết nối GEE, load AOI từ GeoJSON local, kiểm tra Sentinel-1 dataset và render bản đồ tương tác.

In [ ]:
# ─────────────────────────────────────────────
# 1. Khởi tạo GEE
# ─────────────────────────────────────────────
print("=" * 50)
print("SONG HONG SAR MONITORING — Environment Check")
print("=" * 50)

print("✅ GEE initialized: project = crested-library-500309-i2\n")

# ─────────────────────────────────────────────
# 2. Load AOI từ GeoJSON local
# ─────────────────────────────────────────────
aoi = ee.FeatureCollection(aoi_geojson)
aoi_geometry = aoi.geometry()

print(f"✅ AOI loaded from: {aoi_path}")

# In thông tin AOI
aoi_info = aoi.first().getInfo()
props = aoi_info['properties']
print(f"   Name       : {props.get('name', 'N/A')}")
print(f"   Description: {props.get('description', 'N/A')}")
print(f"   Area ~     : {props.get('area_km2_approx', 'N/A')} km²\n")

# ─────────────────────────────────────────────
# 3. Kiểm tra dataset Sentinel-1
# ─────────────────────────────────────────────
s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(aoi_geometry)
        .filterDate('2024-01-01', '2024-12-31')
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        .select(['VV', 'VH']))

count_2024 = s1.size().getInfo()
print(f"✅ Sentinel-1 GRD (IW, DESCENDING, 2024): {count_2024} ảnh\n")

# Metadata ảnh đầu tiên
first_img = s1.first()
first_info = first_img.getInfo()
print("📋 Metadata ảnh đầu tiên:")
print(f"   ID         : {first_info['id']}")
print(f"   Date       : {first_info['properties'].get('system:index', 'N/A')}")
print(f"   Bands      : {[b['id'] for b in first_info['bands']]}")
print(f"   Resolution : {first_info['bands'][0].get('crs_transform', 'N/A')}\n")

# ─────────────────────────────────────────────
# 4. Hiển thị bản đồ tương tác
# ─────────────────────────────────────────────
print("🗺️  Đang khởi tạo bản đồ tương tác (geemap)...")

Map = geemap.Map(center=[21.0, 105.75], zoom=10)
Map.add_basemap('SATELLITE')

# Thêm AOI
Map.addLayer(aoi_geometry, {'color': 'red', 'width': 2}, 'AOI - Song Hong')

# Thêm ảnh VV composite (median tháng 6/2024 - mùa lũ)
s1_june = (ee.ImageCollection('COPERNICUS/S1_GRD')
             .filterBounds(aoi_geometry)
             .filterDate('2024-06-01', '2024-06-30')
             .filter(ee.Filter.eq('instrumentMode', 'IW'))
             .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
             .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
             .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
             .select(['VV', 'VH'])
             .median()
             .clip(aoi_geometry))

vv_vis = {'bands': ['VV'], 'min': -25, 'max': 0, 'palette': ['black', 'white']}
Map.addLayer(s1_june, vv_vis, 'Sentinel-1 VV (Jun 2024)')
Map.centerObject(aoi_geometry, 10)

# Lưu bản đồ ra HTML
out_html = os.path.join(OUT_DIR, 'map_check.html')
os.makedirs(OUT_DIR, exist_ok=True)
Map.to_html(out_html)

print(f"✅ Bản đồ đã lưu: {out_html}")
print("\n" + "=" * 50)
print("✅ ENVIRONMENT CHECK PASSED")
print("=" * 50)


---

## 📊 Phần 2 — Thu thập dữ liệu Sentinel-1 (2015–2024)

Thống kê số lượng ảnh theo năm và tháng, phát hiện gap dữ liệu, lưu kết quả ra CSV và JSON.

In [ ]:
# ─────────────────────────────────────────────
# 1. Khởi tạo
# ─────────────────────────────────────────────
print("=" * 60)
print("SONG HONG SAR — 01: Data Collection (2015–2024)")
print("=" * 60)

# Load AOI
# ─────────────────────────────────────────────
# 2. Truy vấn toàn bộ collection 2015–2024
# ─────────────────────────────────────────────
s1_all = (ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterBounds(aoi_geometry)
            .filterDate('2015-01-01', '2024-12-31')
            .filter(ee.Filter.eq('instrumentMode', 'IW'))
            .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
            .select(['VV', 'VH']))

total = s1_all.size().getInfo()
print(f"\n📊 Tổng số ảnh (2015–2024): {total}")

if total < 500:
    print("⚠️  WARNING: Số ảnh < 500, kiểm tra lại AOI hoặc bộ lọc!")
else:
    print("✅ Đủ dữ liệu cho phân tích chuỗi thời gian 10 năm")

# ─────────────────────────────────────────────
# 3. Thống kê theo năm
# ─────────────────────────────────────────────
print("\n📅 Số ảnh theo năm:")
print(f"{'Năm':<8} {'Số ảnh':<10} {'Tình trạng'}")
print("-" * 35)

years = list(range(2015, 2025))
year_stats = []

for year in years:
    count = (s1_all
             .filter(ee.Filter.calendarRange(year, year, 'year'))
             .size()
             .getInfo())
    status = "✅ OK" if count >= 20 else "⚠️  Thiếu" if count > 0 else "❌ Không có"
    print(f"{year:<8} {count:<10} {status}")
    year_stats.append({'year': year, 'count': count, 'status': status.strip()})

# ─────────────────────────────────────────────
# 4. Thống kê theo tháng (giai đoạn 2020–2024)
# ─────────────────────────────────────────────
print("\n📅 Số ảnh theo tháng (2020–2024 — giai đoạn chính):")
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
month_stats = []

s1_recent = s1_all.filterDate('2020-01-01', '2024-12-31')

for month in range(1, 13):
    count = (s1_recent
             .filter(ee.Filter.calendarRange(month, month, 'month'))
             .size()
             .getInfo())
    bar = '█' * (count // 2)
    print(f"  {month_names[month-1]:<5} {count:>3}  {bar}")
    month_stats.append({'month': month, 'month_name': month_names[month-1], 'count': count})

# ─────────────────────────────────────────────
# 5. Lấy metadata chi tiết (10 ảnh đầu)
# ─────────────────────────────────────────────
print("\n📋 Metadata 5 ảnh đầu tiên:")
first_5 = s1_all.sort('system:time_start').limit(5)
first_5_info = first_5.getInfo()

metadata_list = []
for feat in first_5_info['features']:
    props = feat['properties']
    img_id = props.get('system:index', 'N/A')
    ts = props.get('system:time_start', 0)
    date_str = datetime.utcfromtimestamp(ts / 1000).strftime('%Y-%m-%d') if ts else 'N/A'
    rel_orbit = props.get('relativeOrbitNumber_start', 'N/A')
    print(f"  [{date_str}] ID={img_id} | Orbit={rel_orbit}")
    metadata_list.append({'id': img_id, 'date': date_str, 'relative_orbit': rel_orbit})

# ─────────────────────────────────────────────
# 6. Lưu kết quả
# ─────────────────────────────────────────────

# CSV theo năm
csv_year_path = os.path.join(out_dir, 's1_coverage_by_year.csv')
with open(csv_year_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['year', 'count', 'status'])
    writer.writeheader()
    writer.writerows(year_stats)
print(f"\n✅ Lưu: {csv_year_path}")

# CSV theo tháng
csv_month_path = os.path.join(out_dir, 's1_coverage_by_month.csv')
with open(csv_month_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['month', 'month_name', 'count'])
    writer.writeheader()
    writer.writerows(month_stats)
print(f"✅ Lưu: {csv_month_path}")

# JSON metadata
json_path = os.path.join(out_dir, 's1_metadata.json')
report = {
    'generated_at': datetime.utcnow().isoformat() + 'Z',
    'project': 'crested-library-500309-i2',
    'total_images_2015_2024': total,
    'filter_params': {
        'instrument_mode': 'IW',
        'orbit_direction': 'DESCENDING',
        'polarisation': ['VV', 'VH']
    },
    'year_stats': year_stats,
    'month_stats_2020_2024': month_stats,
    'sample_metadata': metadata_list
}
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(f"✅ Lưu: {json_path}")

print("\n" + "=" * 60)
print(f"✅ DATA COLLECTION DONE — {total} ảnh sẵn sàng")
print("=" * 60)


---

## ⚙️ Phần 3 — Tiền xử lý & Tính toán đặc trưng SAR

Lọc speckle (Refined Lee Filter), tính đặc trưng VV / VH / VV-VH ratio, tạo composite tháng.

In [ ]:
# ─────────────────────────────────────────────
# 1. Khởi tạo
# ─────────────────────────────────────────────
print("=" * 60)
print("SONG HONG SAR — 02: Preprocessing & Feature Extraction")
print("=" * 60)

# Load AOI
# ─────────────────────────────────────────────
# 2. Hàm tiền xử lý
# ─────────────────────────────────────────────

def apply_speckle_filter(image):
    """
    Refined Lee Speckle Filter (3×3 kernel).
    Hoạt động trên linear scale, chuyển đổi qua lại từ dB.
    """
    band_names = image.bandNames()

    # Convert từ dB → linear
    img_linear = ee.Image(10.0).pow(image.divide(10.0))

    kernel = ee.Kernel.square(radius=1)  # 3×3

    # Tính mean và variance cục bộ
    mean_img = img_linear.reduceNeighborhood(
        reducer=ee.Reducer.mean(),
        kernel=kernel
    )
    variance_img = img_linear.reduceNeighborhood(
        reducer=ee.Reducer.variance(),
        kernel=kernel
    )

    # Hệ số biến động (coefficient of variation)
    img_cv = variance_img.divide(mean_img.pow(2))

    # Trọng số Lee: w = 1 - ENL_variance / img_cv
    # ENL (Equivalent Number of Looks) ≈ 4.9 cho Sentinel-1 IW
    ENL = 4.9
    ENL_variance = ee.Image(1.0 / ENL)
    weight = ee.Image(1.0).subtract(
        ENL_variance.divide(img_cv.add(ENL_variance))
    ).max(0).min(1)

    # Filtered = mean + weight * (pixel - mean)
    filtered_linear = mean_img.add(
        weight.multiply(img_linear.subtract(mean_img))
    )

    # Convert lại → dB
    filtered_db = filtered_linear.log10().multiply(10.0)
    return filtered_db.rename(band_names).copyProperties(image, image.propertyNames())


def compute_features(image):
    """
    Tính 3 đặc trưng SAR:
      - VV  : backscatter dB (sau lọc speckle)
      - VH  : backscatter dB (sau lọc speckle)
      - VV_VH_ratio : tỉ số VV/VH (linear → dB) = VV - VH
    """
    vv = image.select('VV')
    vh = image.select('VH')

    # Ratio (dB) = VV(dB) - VH(dB)  [= log10(VV_linear/VH_linear) * 10]
    ratio = vv.subtract(vh).rename('VV_VH_ratio')

    return image.addBands(ratio).select(['VV', 'VH', 'VV_VH_ratio'])


def preprocess_image(image):
    """Pipeline đầy đủ cho 1 ảnh."""
    clipped = image.clip(aoi_geometry)
    filtered = apply_speckle_filter(clipped)
    features = compute_features(filtered)
    return features.copyProperties(image, ['system:time_start', 'system:index'])


# ─────────────────────────────────────────────
# 3. Áp dụng pipeline lên collection 2015–2024
# ─────────────────────────────────────────────
s1_raw = (ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterBounds(aoi_geometry)
            .filterDate('2015-01-01', '2024-12-31')
            .filter(ee.Filter.eq('instrumentMode', 'IW'))
            .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
            .select(['VV', 'VH']))

s1_processed = s1_raw.map(preprocess_image)
print(f"\n✅ Pipeline áp dụng: {s1_processed.size().getInfo()} ảnh đã xử lý")

# ─────────────────────────────────────────────
# 4. Hàm tạo composite tháng
# ─────────────────────────────────────────────

def monthly_composite(year, month):
    """Tạo composite median cho 1 tháng, clip theo AOI."""
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')
    composite = (s1_processed
                 .filterDate(start, end)
                 .median()
                 .clip(aoi_geometry))
    return composite.set({
        'year': year,
        'month': month,
        'system:time_start': start.millis(),
        'system:index': ee.String(ee.Number(year).format('%04d'))
                          .cat('_')
                          .cat(ee.String(ee.Number(month).format('%02d')))
    })


# ─────────────────────────────────────────────
# 5. Kiểm tra giá trị đặc trưng (mẫu tháng 1/2024 vs 8/2024)
# ─────────────────────────────────────────────
print("\n📊 Kiểm tra đặc trưng — Tháng 1/2024 (mùa khô) vs 8/2024 (mùa lũ):")

# Điểm kiểm tra: mặt nước sông (Long Bien)
water_point = ee.Geometry.Point([105.8650, 21.0420])
# Điểm kiểm tra: đất / bờ sông
land_point = ee.Geometry.Point([105.8700, 21.0500])

for year, month, label in [(2024, 1, 'Mùa khô (T1/2024)'), (2024, 8, 'Mùa lũ  (T8/2024)')]:
    comp = monthly_composite(year, month)
    scale = 10  # Sentinel-1 resolution 10m

    water_val = comp.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=water_point.buffer(100),
        scale=scale
    ).getInfo()

    land_val = comp.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=land_point.buffer(100),
        scale=scale
    ).getInfo()

    print(f"\n  [{label}]")
    if water_val.get('VV') is not None:
        print(f"    Điểm NƯỚC  → VV={water_val.get('VV', 'N/A'):.2f} dB | "
              f"VH={water_val.get('VH', 'N/A'):.2f} dB | "
              f"Ratio={water_val.get('VV_VH_ratio', 'N/A'):.2f} dB")
        print(f"    Điểm ĐẤT   → VV={land_val.get('VV', 'N/A'):.2f} dB | "
              f"VH={land_val.get('VH', 'N/A'):.2f} dB | "
              f"Ratio={land_val.get('VV_VH_ratio', 'N/A'):.2f} dB")

        # Kiểm tra ngưỡng
        vv_water = water_val.get('VV', 0)
        if vv_water < -15:
            print(f"    ✅ VV mặt nước = {vv_water:.2f} dB < -15 dB (đạt tiêu chí)")
        else:
            print(f"    ⚠️  VV mặt nước = {vv_water:.2f} dB — kiểm tra lại điểm tham chiếu")
    else:
        print(f"    ⚠️  Không có dữ liệu cho tháng này tại điểm kiểm tra")

# ─────────────────────────────────────────────
# 6. Lưu thông tin pipeline
# ─────────────────────────────────────────────
pipeline_info = {
    'generated_at': datetime.utcnow().isoformat() + 'Z',
    'pipeline_steps': [
        '1. Filter: IW mode, DESCENDING, VV+VH bands',
        '2. Clip to AOI',
        '3. Refined Lee Speckle Filter (3×3, ENL=4.9)',
        '4. Feature: VV_VH_ratio = VV(dB) - VH(dB)',
        '5. Monthly median composite'
    ],
    'output_bands': ['VV', 'VH', 'VV_VH_ratio'],
    'expected_values': {
        'water_VV_dB': '< -15',
        'land_VV_dB': '> -10',
        'sandbars_VV_dB': '-15 to -10 (intermediate)'
    }
}

stats_path = os.path.join(out_dir, 'preprocessing_stats.json')
with open(stats_path, 'w', encoding='utf-8') as f:
    json.dump(pipeline_info, f, indent=2, ensure_ascii=False)

print(f"\n✅ Pipeline info lưu: {stats_path}")
print("\n" + "=" * 60)
print("✅ PREPROCESSING PIPELINE READY")
print("   Dùng monthly_composite(year, month) để tạo ảnh bất kỳ")
print("=" * 60)


---

## 🗺️ Phần 4 — Kiểm tra trực quan

So sánh composite mùa khô vs mùa lũ, hiển thị 3 đặc trưng SAR, đối chiếu với Sentinel-2 RGB.

In [ ]:
import sys

# Import preprocessing pipeline từ script 02
sys.path.insert(0, os.path.dirname(__file__))

# ─────────────────────────────────────────────
# 1. Khởi tạo
# ─────────────────────────────────────────────
print("=" * 60)
print("SONG HONG SAR — 03: Visualization Check")
print("=" * 60)

# Load AOI
# ─────────────────────────────────────────────
# 2. Copy pipeline từ script 02 (standalone)
# ─────────────────────────────────────────────

def apply_speckle_filter(image):
    band_names = image.bandNames()
    img_linear = ee.Image(10.0).pow(image.divide(10.0))
    kernel = ee.Kernel.square(radius=1)
    mean_img = img_linear.reduceNeighborhood(ee.Reducer.mean(), kernel)
    variance_img = img_linear.reduceNeighborhood(ee.Reducer.variance(), kernel)
    img_cv = variance_img.divide(mean_img.pow(2))
    ENL_variance = ee.Image(1.0 / 4.9)
    weight = ee.Image(1.0).subtract(ENL_variance.divide(img_cv.add(ENL_variance))).max(0).min(1)
    filtered_linear = mean_img.add(weight.multiply(img_linear.subtract(mean_img)))
    filtered_db = filtered_linear.log10().multiply(10.0)
    return filtered_db.rename(band_names).copyProperties(image, image.propertyNames())

def compute_features(image):
    vv = image.select('VV')
    vh = image.select('VH')
    ratio = vv.subtract(vh).rename('VV_VH_ratio')
    return image.addBands(ratio).select(['VV', 'VH', 'VV_VH_ratio'])

def preprocess_image(image):
    clipped = image.clip(aoi_geometry)
    filtered = apply_speckle_filter(clipped)
    return compute_features(filtered).copyProperties(image, ['system:time_start', 'system:index'])

s1_processed = (ee.ImageCollection('COPERNICUS/S1_GRD')
                  .filterBounds(aoi_geometry)
                  .filterDate('2020-01-01', '2024-12-31')
                  .filter(ee.Filter.eq('instrumentMode', 'IW'))
                  .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
                  .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
                  .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
                  .select(['VV', 'VH'])
                  .map(preprocess_image))

# ─────────────────────────────────────────────
# 3. Tạo composite mùa khô và mùa lũ
# ─────────────────────────────────────────────
def season_composite(months, years=(2020, 2024)):
    """Composite median cho nhiều năm, chọn theo tháng."""
    month_filter = ee.Filter.calendarRange(months[0], months[-1], 'month')
    return (s1_processed
            .filterDate(f'{years[0]}-01-01', f'{years[1]}-12-31')
            .filter(month_filter)
            .median()
            .clip(aoi_geometry))

# Mùa khô: tháng 11–4 (ít mưa, mực nước thấp)
dry_composite   = season_composite([11, 4])  # Nov–Apr (sẽ dùng calendarRange riêng)
# Cách đơn giản hơn:
dry_composite = (s1_processed
                 .filterDate('2020-01-01', '2024-12-31')
                 .filter(ee.Filter.Or(
                     ee.Filter.calendarRange(11, 12, 'month'),
                     ee.Filter.calendarRange(1, 4, 'month')
                 ))
                 .median()
                 .clip(aoi_geometry))

# Mùa lũ: tháng 6–9 (mực nước cao, bãi bồi ngập)
wet_composite = (s1_processed
                 .filterDate('2020-01-01', '2024-12-31')
                 .filter(ee.Filter.calendarRange(6, 9, 'month'))
                 .median()
                 .clip(aoi_geometry))

print("\n✅ Composite mùa khô và mùa lũ đã sẵn sàng")

# ─────────────────────────────────────────────
# 4. Cài đặt hiển thị màu
# ─────────────────────────────────────────────
vis_vv = {
    'bands': ['VV'], 'min': -25, 'max': 0,
    'palette': ['#000033', '#003399', '#0099FF', '#FFFFFF']
}
vis_vh = {
    'bands': ['VH'], 'min': -30, 'max': -5,
    'palette': ['#000033', '#003399', '#0099FF', '#FFFFFF']
}
vis_ratio = {
    'bands': ['VV_VH_ratio'], 'min': 2, 'max': 15,
    'palette': ['#440154', '#31688E', '#35B779', '#FDE725']  # Viridis
}

# ─────────────────────────────────────────────
# 5. Bản đồ 1: So sánh mùa khô vs mùa lũ (VV band)
# ─────────────────────────────────────────────
print("\n🗺️  Tạo bản đồ 1: So sánh mùa khô vs mùa lũ...")

Map1 = geemap.Map(center=[21.0, 105.75], zoom=10)
Map1.add_basemap('SATELLITE')
Map1.addLayer(aoi_geometry, {'color': 'yellow', 'width': 1}, 'AOI')
Map1.addLayer(dry_composite,  vis_vv, 'VV — Mùa khô (Nov–Apr)')
Map1.addLayer(wet_composite,  vis_vv, 'VV — Mùa lũ (Jun–Sep)')

# Điểm tham chiếu
key_points = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([105.8650, 21.0420]), {'name': 'Long Bien - Song'}),
    ee.Feature(ee.Geometry.Point([105.8500, 21.0800]), {'name': 'Nhat Tan - Song'}),
    ee.Feature(ee.Geometry.Point([105.8800, 20.9900]), {'name': 'Vinh Tuy - Song'}),
])
Map1.addLayer(key_points, {'color': 'red'}, 'Khu vực trọng điểm')

html1 = os.path.join(out_dir, 'visualization_dry_wet.html')
Map1.to_html(html1)
print(f"   ✅ Lưu: {html1}")

# ─────────────────────────────────────────────
# 6. Bản đồ 2: 3 đặc trưng (VV, VH, Ratio) — tháng 1/2024
# ─────────────────────────────────────────────
print("\n🗺️  Tạo bản đồ 2: Ba đặc trưng SAR (tháng 1/2024)...")

jan2024 = (s1_processed
           .filterDate('2024-01-01', '2024-01-31')
           .median()
           .clip(aoi_geometry))

Map2 = geemap.Map(center=[21.0, 105.75], zoom=10)
Map2.add_basemap('SATELLITE')
Map2.addLayer(aoi_geometry, {'color': 'yellow', 'width': 1}, 'AOI')
Map2.addLayer(jan2024, vis_vv,    'VV band (dB)')
Map2.addLayer(jan2024, vis_vh,    'VH band (dB)')
Map2.addLayer(jan2024, vis_ratio, 'VV/VH Ratio (dB)')

html2 = os.path.join(out_dir, 'visualization_features.html')
Map2.to_html(html2)
print(f"   ✅ Lưu: {html2}")

# ─────────────────────────────────────────────
# 7. Thêm Sentinel-2 RGB để đối chiếu
# ─────────────────────────────────────────────
print("\n🛰️  Thêm Sentinel-2 RGB để đối chiếu...")

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi_geometry)
        .filterDate('2024-01-01', '2024-01-31')
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .select(['B4', 'B3', 'B2'])  # RGB
        .median()
        .clip(aoi_geometry))

Map2.addLayer(s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'Sentinel-2 RGB (Jan 2024)')
Map2.to_html(html2)  # Ghi đè để cập nhật
print(f"   ✅ Cập nhật: {html2}")

print("\n" + "=" * 60)
print("✅ VISUALIZATION DONE")
print(f"   Mở bằng trình duyệt:")
print(f"   - {html1}")
print(f"   - {html2}")
print("=" * 60)


---

## 📤 Phần 5 — Export GeoTIFF mẫu lên Google Drive

Export composite tháng 1/2020 (khô), tháng 8/2020 (lũ) và annual 2024 để kiểm tra ngoài GEE.

In [ ]:
# ─────────────────────────────────────────────
# 1. Khởi tạo
# ─────────────────────────────────────────────
print("=" * 60)
print("SONG HONG SAR — 04: Export Sample Composites")
print("=" * 60)

# Load AOI
# ─────────────────────────────────────────────
# 2. Pipeline tiền xử lý (standalone)
# ─────────────────────────────────────────────

def apply_speckle_filter(image):
    band_names = image.bandNames()
    img_linear = ee.Image(10.0).pow(image.divide(10.0))
    kernel = ee.Kernel.square(radius=1)
    mean_img = img_linear.reduceNeighborhood(ee.Reducer.mean(), kernel)
    variance_img = img_linear.reduceNeighborhood(ee.Reducer.variance(), kernel)
    img_cv = variance_img.divide(mean_img.pow(2))
    ENL_variance = ee.Image(1.0 / 4.9)
    weight = ee.Image(1.0).subtract(ENL_variance.divide(img_cv.add(ENL_variance))).max(0).min(1)
    filtered_linear = mean_img.add(weight.multiply(img_linear.subtract(mean_img)))
    filtered_db = filtered_linear.log10().multiply(10.0)
    return filtered_db.rename(band_names).copyProperties(image, image.propertyNames())

def compute_features(image):
    vv = image.select('VV')
    vh = image.select('VH')
    ratio = vv.subtract(vh).rename('VV_VH_ratio')
    return image.addBands(ratio).select(['VV', 'VH', 'VV_VH_ratio'])

def preprocess_image(image):
    return compute_features(
        apply_speckle_filter(image.clip(aoi_geometry))
    ).copyProperties(image, ['system:time_start', 'system:index'])

def get_monthly_composite(year, month):
    """Trả về composite median của 1 tháng cụ thể."""
    start = f'{year}-{month:02d}-01'
    end_date = ee.Date.fromYMD(year, month, 1).advance(1, 'month')
    end = end_date.format('YYYY-MM-dd').getInfo()

    composite = (ee.ImageCollection('COPERNICUS/S1_GRD')
                   .filterBounds(aoi_geometry)
                   .filterDate(start, end)
                   .filter(ee.Filter.eq('instrumentMode', 'IW'))
                   .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
                   .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
                   .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
                   .select(['VV', 'VH'])
                   .map(preprocess_image)
                   .median()
                   .clip(aoi_geometry))
    return composite

def get_annual_composite(year):
    """Composite median toàn năm."""
    composite = (ee.ImageCollection('COPERNICUS/S1_GRD')
                   .filterBounds(aoi_geometry)
                   .filterDate(f'{year}-01-01', f'{year}-12-31')
                   .filter(ee.Filter.eq('instrumentMode', 'IW'))
                   .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
                   .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
                   .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
                   .select(['VV', 'VH'])
                   .map(preprocess_image)
                   .median()
                   .clip(aoi_geometry))
    return composite

# ─────────────────────────────────────────────
# 3. Tạo các ảnh cần export
# ─────────────────────────────────────────────
exports = [
    {
        'image': get_monthly_composite(2020, 1),
        'description': 'SongHong_S1_Composite_2020_01_dry',
        'label': 'Tháng 1/2020 — Mùa khô'
    },
    {
        'image': get_monthly_composite(2020, 8),
        'description': 'SongHong_S1_Composite_2020_08_wet',
        'label': 'Tháng 8/2020 — Mùa lũ'
    },
    {
        'image': get_annual_composite(2024),
        'description': 'SongHong_S1_Annual_2024',
        'label': 'Composite cả năm 2024'
    }
]

# ─────────────────────────────────────────────
# 4. Submit export tasks
# ─────────────────────────────────────────────
print(f"\n📤 Submitting {len(exports)} export tasks...\n")

task_ids = []
for exp in exports:
    task = ee.batch.Export.image.toDrive(
        image=exp['image'],
        description=exp['description'],
        folder='SongHong_SAR',             # Tạo thư mục này trên Google Drive
        fileNamePrefix=exp['description'],
        region=aoi_geometry,
        scale=10,                           # 10m resolution (Sentinel-1 native)
        crs='EPSG:32648',                   # UTM Zone 48N — phù hợp với Hà Nội
        maxPixels=1e10,
        fileFormat='GeoTIFF'
    )
    task.start()
    task_id = task.id
    task_ids.append({'label': exp['label'], 'id': task_id, 'description': exp['description']})
    print(f"  ✅ [{exp['label']}]")
    print(f"     Task ID: {task_id}")
    print(f"     File   : SongHong_SAR/{exp['description']}.tif\n")

# ─────────────────────────────────────────────
# 5. Lưu task IDs để theo dõi
# ─────────────────────────────────────────────
tasks_log = {
    'submitted_at': datetime.utcnow().isoformat() + 'Z',
    'tasks': task_ids,
    'check_status': 'https://code.earthengine.google.com/tasks',
    'drive_folder': 'SongHong_SAR',
    'verification': {
        'open_with': 'QGIS hoặc Python (rasterio)',
        'expected_bands': ['VV', 'VH', 'VV_VH_ratio'],
        'expected_crs': 'EPSG:32648',
        'expected_scale_m': 10,
        'VV_water_expected': '< -15 dB',
        'VV_land_expected': '> -10 dB'
    }
}

log_path = os.path.join(out_dir, 'export_tasks.json')
with open(log_path, 'w', encoding='utf-8') as f:
    json.dump(tasks_log, f, indent=2, ensure_ascii=False)
print(f"✅ Task log lưu: {log_path}")

print("\n" + "=" * 60)
print("✅ EXPORT SUBMITTED")
print("   Theo dõi tại: https://code.earthengine.google.com/tasks")
print("   Thường hoàn thành trong 5–15 phút")
print("=" * 60)
